# Assignment 4 - Hyperparameter optimization

## Assignment objective
Compare the lecture baseline **dtc3** with new Decision Tree models by changing:
- `max_depth`
- `min_samples_leaf`

Evaluation used:
- Precision
- Recall
- F1
- ROC AUC
- `precision@3000`
- `recall@3000`

In [ ]:

import pandas as pd
import numpy as np
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

DATA_PATH = ".../Churn_Banking_Modeling_ENG.csv"
TOP_K = 3000
RANDOM_STATE = 42

features = [
    "amt_cust_value", "flag_online_acc_opening", "flag_mult_account_ownership",
    "num_age", "num_year_first_account", "amt_pricing_fee",
    "amt_transfer_vs_competitors", "amt_tranfers_vs_no_competitors",
    "num_existing_services", "flag_salary_deposit", "amt_credit_card_spending",
    "amt_debit_card_spending", "num_website_access_count", "num_transactions_count",
    "num_trading_activities_count", "str_change_num_utilities", "flag_mortgage",
    "flag_loan", "flag_internal_tranfers", "flag_request_info_closure",
    "flag_loyalty_program_enrol", "flag_call_center_contact",
    "flag_salary_deposit_variation", "num_loyalty_points", "amt_current_liquidity",
    "amt_current_managed", "amt_current_administered", "amt_6m_current_liquidity",
    "amt_6m_current_managed", "amt_6m_current_administered",
    "flag_outgoing_sec_tranfer", "flag_card_rejection", "flag_loan_rejection",
    "flag_deactivation_rid"
]

# Load data and prepare target

df = pd.read_csv(DATA_PATH)
df = df.rename(columns={"flag_request_closure": "Target"})
df["Target"] = df["Target"].map({"si": 1, "no": 0})
df_model = df[["Target"] + features].copy().fillna(0)

X = df_model[features]
y = df_model["Target"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

# Undersampling only on train set, as done in class
rus = RandomUnderSampler(sampling_strategy=0.20, random_state=RANDOM_STATE)
X_train_bal, y_train_bal = rus.fit_resample(X_train, y_train)

print("Train shape:", X_train.shape)
print("Balanced train shape:", X_train_bal.shape)
print("Test shape:", X_test.shape)
print("Train churn rate:", round(y_train.mean(), 4))
print("Test churn rate:", round(y_test.mean(), 4))
print("Balanced train churn rate:", round(pd.Series(y_train_bal).mean(), 4))


Train shape: (264158, 34)
Balanced train shape: (8328, 34)
Test shape: (113211, 34)
Train churn rate: 0.0053
Test churn rate: 0.0053
Balanced train churn rate: 0.1667


In [2]:

def top_k_metrics(model, X_test, y_test, k=3000):
    scores = model.predict_proba(X_test)[:, 1]
    ranking = pd.DataFrame({"y_true": y_test.to_numpy(), "score": scores}).sort_values("score", ascending=False)
    top = ranking.head(min(k, len(ranking)))
    positives = int(top["y_true"].sum())
    total_positives = int(ranking["y_true"].sum())
    return positives / len(top), positives / total_positives


def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]
    p_at_k, r_at_k = top_k_metrics(model, X_test, y_test, k=TOP_K)
    return {
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score),
        "precision_at_3000": p_at_k,
        "recall_at_3000": r_at_k,
        "tree_depth": model.get_depth(),
        "n_leaves": model.get_n_leaves(),
    }


In [3]:

# Baseline dtc3 from the lecture notebook
baseline = DecisionTreeClassifier(max_depth=4, min_samples_leaf=1, random_state=RANDOM_STATE)
baseline.fit(X_train_bal, y_train_bal)

baseline_results = evaluate_model(baseline, X_test, y_test)
pd.DataFrame([baseline_results], index=["baseline_dtc3"]).T


,baseline_dtc3
precision,0.035453
recall,0.448739
f1,0.065715
roc_auc,0.836635
precision_at_3000,0.043000
recall_at_3000,0.216807
tree_depth,4.000000
n_leaves,15.000000


In [4]:

# Hyperparameter experiments
max_depth_values = [2, 3, 4, 5, 6, 8, 10]
min_samples_leaf_values = [1, 5, 10, 25, 50, 100]

results = []
for md in max_depth_values:
    for msl in min_samples_leaf_values:
        model = DecisionTreeClassifier(max_depth=md, min_samples_leaf=msl, random_state=RANDOM_STATE)
        model.fit(X_train_bal, y_train_bal)
        metrics = evaluate_model(model, X_test, y_test)
        metrics["max_depth"] = md
        metrics["min_samples_leaf"] = msl
        results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(["precision_at_3000", "recall_at_3000", "f1"], ascending=False)
results_df.head(10)


,precision,recall,f1,roc_auc,precision_at_3000,recall_at_3000,tree_depth,n_leaves,max_depth,min_samples_leaf
24,0.047424,0.526050,0.087005,0.860900,0.064333,0.324370,6,47,6,1
41,0.055098,0.359664,0.095557,0.853815,0.063667,0.321008,10,46,10,100
34,0.043116,0.478992,0.079112,0.859321,0.063333,0.319328,8,66,8,50
35,0.055098,0.359664,0.095557,0.855479,0.063000,0.317647,8,41,8,100
25,0.047848,0.519328,0.087622,0.860394,0.062667,0.315966,6,47,6,5
33,0.057458,0.403361,0.100587,0.862900,0.062333,0.314286,8,82,8,25
40,0.042370,0.447059,0.077404,0.856376,0.062000,0.312605,10,91,10,50
39,0.047542,0.442017,0.085850,0.853374,0.061333,0.309244,10,123,10,25
18,0.036004,0.514286,0.067297,0.854438,0.061000,0.307563,5,28,5,1
37,0.042399,0.502521,0.078201,0.830540,0.060667,0.305882,10,214,10,5


In [5]:

# Compare baseline with the best model by precision@3000
best_model_row = results_df.iloc[0]
comparison = pd.DataFrame([
    {"model": "baseline_dtc3", "max_depth": 4, "min_samples_leaf": 1, **baseline_results},
    {"model": "best_model", **best_model_row.to_dict()}
])
comparison


,model,max_depth,min_samples_leaf,precision,recall,f1,roc_auc,precision_at_3000,recall_at_3000,tree_depth,n_leaves
0,baseline_dtc3,4.0,1.0,0.035453,0.448739,0.065715,0.836635,0.043000,0.216807,4.0,15.0
1,best_model,6.0,1.0,0.047424,0.526050,0.087005,0.860900,0.064333,0.324370,6.0,47.0


In [6]:
# Save results
output_path = "/mnt/data/assignment4_simple_results.csv"
results_df.to_csv(output_path, index=False)
print("Results saved to:", output_path)

print("\nBest model by precision@3000")
print(best_model_row[["max_depth", "min_samples_leaf", "precision_at_3000", "recall_at_3000", "f1", "roc_auc"]])


Results saved to: /mnt/data/assignment4_simple_results.csv

Best model by precision@3000
max_depth            6.000000
min_samples_leaf     1.000000
precision_at_3000    0.064333
recall_at_3000       0.324370
f1                   0.087005
roc_auc              0.860900
Name: 24, dtype: float64


## Final comments for the assignment

### Objective
The goal of this assignment is to evaluate the impact of hyperparameter tuning on a Decision Tree classifier applied to the banking churn dataset.

The two hyperparameters analyzed are:
- `max_depth`
- `min_samples_leaf`

The purpose is not only to improve performance, but also to show the ability to run experiments, compare models, and comment the results.

### Method
Here the steps:
1. loaded the banking churn dataset
2. selected the same main numerical variables used during the class
3. transformed the target from `si/no` to `1/0` (as did in class)
4. handled missing values with `fillna(0)`
5. split the data into train and test sets 
6. applied `RandomUnderSampler` on the training set to manage class imbalance
7. rebuilt the lecture baseline model `dtc3`
8. trained new Decision Tree models by changing `max_depth` and `min_samples_leaf`

### Evaluation setup
To compare models, I used the folloqing metrics:
- Precision
- Recall
- F1-score
- ROC AUC
- `precision@3000`
- `recall@3000`

The metrics `precision@3000` and `recall@3000` are useful because they simulate a business case in which the bank contacts the 3000 customers with the highest predicted churn score.

### Baseline
The baseline model is the lecture model `dtc3`:
- `max_depth = 4`
- `min_samples_leaf = 1`

This model is the benchmark used to evaluate whether the new models perform better or worse.

### Main findings
From the experiments:
- the baseline `dtc3` gives a reasonable starting point
- increasing `max_depth` usually helps the model capture more patterns and can improve ranking performance
- very restrictive values of `min_samples_leaf` make the tree simpler, but can reduce the ability to identify churners
- deeper trees can improve recall, but they may also risk overfitting

In my results, one of the best models for business-oriented targeting is a tree with a slightly larger `max_depth` than the baseline. This suggests that allowing the tree to learn a bit more structure can improve identification of the most relevant customers.

### Conclusion
Overall, hyperparameter tuning has a clear effect on model performance, and the best configuration depends on the metric chosen. For a churn campaign, `precision@3000` and `recall@3000` are especially useful because they focus on the customers that the business would actually target.


## Declaration

I hereby declare that this assignment is my own work and that all references are in line with the course material and the assignment instructions.
